# Robustness to Trotter repetitions

**Normalized Ising instances, $\beta=4$, $n=3,\ldots,10$, 100 instances.**

The 300-task balanced SLURM array evaluates ten Trotter repetitions at four $(\gamma,t)$ midpoint grids. This notebook only reads the merged spectral gaps; no extra proposal-error calculations are performed.

## 1. Load merged spectral gaps

In [ ]:
from pathlib import Path
import h5py
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({"savefig.bbox": "tight", "axes.spines.top": False,
                     "axes.spines.right": False})
HERE = Path.cwd()
FILE = HERE / "spectral_gaps_trotter_merged.hdf5"
with h5py.File(FILE) as f:
    RS = np.asarray(f.attrs["rs"], int)
    KS = np.asarray(f.attrs["ks"], int)
    BETA = float(f.attrs["beta"])
    NS, COUNT = range(3, 11), 100
    gaps = {k: {n: np.array([f[f"k={k}/{n}/{i}"][:]
                           if f"k={k}/{n}/{i}" in f else np.full(len(RS), np.nan)
                           for i in range(COUNT)]) for n in NS} for k in KS}

# Optional original exact-proposal gaps, computed independently by monaqa2.
candidates = [HERE.parent / "robustness_model_normalization" / "sk_variants" / "spectral_gaps_models_merged.hdf5",
              HERE.parent / "sk_variants" / "spectral_gaps_models_merged.hdf5"]
exact = {n: np.full(COUNT, np.nan) for n in NS}
for path in candidates:
    if path.exists():
        with h5py.File(path) as f:
            exact = {n: np.array([f[f"layden/normalized/{n}/{i}"][()]
                                  if f"layden/normalized/{n}/{i}" in f else np.nan
                                  for i in range(COUNT)]) for n in NS}
        print("Exact reference:", path)
        break
print(f"beta={BETA:g}, r={list(RS)}, grids={[2**int(k) for k in KS]}")

## 2. Completeness table

In [ ]:
print(f"{'grid':>7} {'n':>3} {'completed':>11} {'total':>6}  {'median gap, r=50':>18} {'median gap, r=100':>19}")
for k in KS:
    for n in NS:
        a = gaps[k][n]
        i50, i100 = np.flatnonzero(RS == 50)[0], np.flatnonzero(RS == 100)[0]
        med = lambda v: f"{np.median(v[np.isfinite(v) & (v>0)]):.4g}" if np.any(np.isfinite(v) & (v>0)) else "--"
        print(f"{2**int(k):>3}x{2**int(k):<3} {n:>3} {np.isfinite(a).sum():>11} {a.size:>6}  {med(a[:, i50]):>18} {med(a[:, i100]):>19}")

## 3. Spectral gap vs Trotter repetitions

Mean ± standard deviation across instances, with the exact-evolution mean (dashed) when available.

In [ ]:
colors_n = {3: "#222222", 5: "#004488", 8: "#EE7733", 10: "#AA3377"}
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)
for ax, k in zip(axes.flat, KS):
    for n, color in colors_n.items():
        a = gaps[k][n]
        if not np.any(np.isfinite(a)):
            continue
        mean, std = np.nanmean(a, axis=0), np.nanstd(a, axis=0)
        ax.plot(RS, mean, '-o', ms=3.3, lw=1.6, color=color, label=f'$n={n}$')
        ax.fill_between(RS, np.maximum(mean-std, 1e-16), mean+std,
                        color=color, alpha=.16, linewidth=0)
        if np.isfinite(exact[n]).any():
            ax.axhline(np.nanmean(exact[n]), color=color, ls='--', lw=.9, alpha=.55)
    ax.set(xscale='log', yscale='log', title=rf'Grid ${2**int(k)}\times{2**int(k)}$')
    ax.set_xticks(RS, [str(x) for x in RS], rotation=45)
    ax.grid(False)
for ax in axes[-1]: ax.set_xlabel('Trotter repetitions $r$')
for ax in axes[:, 0]: ax.set_ylabel(r'Spectral gap $\delta$')
axes[0, 0].legend(frameon=False, loc='best')
fig.suptitle(rf'Second-order Trotter robustness, $\beta={BETA:g}$')
fig.tight_layout()
plt.show()

## 4. Grid-discretization sensitivity

Compare grid resolutions at fixed system size using the same Trotter counts.

In [ ]:
colors_k = {1: "#222222", 2: "#004488", 3: "#EE7733", 4: "#AA3377"}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), sharey=True)
for ax, n in zip(axes, (6, 10)):
    for k, color in colors_k.items():
        data = gaps[k][n]
        if not np.isfinite(data).any(): continue
        mean, std = np.nanmean(data, axis=0), np.nanstd(data, axis=0)
        ax.plot(RS, mean, '-o', ms=3.3, lw=1.6, color=color,
                label=rf'${2**k}\times{2**k}$')
        ax.fill_between(RS, np.maximum(mean-std, 1e-16), mean+std,
                        color=color, alpha=.14, linewidth=0)
    if np.isfinite(exact[n]).any():
        ax.axhline(np.nanmean(exact[n]), color='0.4', ls='--', label='Exact')
    ax.set(xscale='log', yscale='log', title=f'$n={n}$', xlabel='Trotter repetitions $r$')
    ax.set_xticks(RS, [str(x) for x in RS], rotation=45)
axes[0].set_ylabel(r'Spectral gap $\delta$')
axes[1].legend(frameon=False, loc='best')
fig.tight_layout()
plt.show()

## 5. Paired spectral-gap ratios

Compare Trotter and exact spectral gaps for the same instances (when the external exact-result file is available).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.5), sharey=True)
for ax, n in zip(axes, (6, 10)):
    k = 4
    positions, values = [], []
    if np.isfinite(exact[n]).any():
        for j, r in enumerate(RS):
            ratio = np.divide(gaps[k][n][:, j], exact[n],
                              out=np.full(COUNT, np.nan), where=exact[n]>0)
            ratio = ratio[np.isfinite(ratio) & (ratio>0)]
            if len(ratio) >= 2 and not np.allclose(ratio, ratio[0]):
                positions.append(j)
                values.append(ratio)
    if values:
        violin = ax.violinplot(values, positions=positions, showmedians=True,
                               showextrema=False, widths=.72)
        for body in violin['bodies']:
            body.set_facecolor(colors_k[k])
            body.set_edgecolor(colors_k[k])
            body.set_alpha(.42)
        violin['cmedians'].set_color('#222222')
    else:
        ax.text(.5, .5, 'Insufficient paired data', ha='center', va='center',
                transform=ax.transAxes)
    ax.axhline(1, lw=.9, ls='--', color='0.45')
    ax.set(xticks=range(len(RS)), xticklabels=RS, title=rf'$n={n}$, grid $16\times16$',
           xlabel='Trotter repetitions $r$')
    ax.tick_params(axis='x', rotation=45)
    ax.set_yscale('log')
axes[0].set_ylabel(r'Gap ratio $\delta_r/\delta_{\mathrm{exact}}$')
fig.tight_layout()
plt.show()